In [13]:
# ============================================================
# Easy Read LLM 서비스 최종 Colab 실행 코드
# 모델: gpt-3.5-turbo
# 실행 환경: Google Colab /content
# ============================================================

# 1. 필요한 패키지 설치
!pip install -q streamlit openai pandas plotly

# 2. app.py 파일 생성
app_code = r'''
import streamlit as st
from openai import OpenAI
import pandas as pd
import plotly.graph_objects as go
import json
import re
import os

# ------------------------------------------------------------
# 기본 설정
# ------------------------------------------------------------
st.set_page_config(
    page_title="Easy Read 변환 서비스",
    page_icon="📖",
    layout="wide"
)

MODEL_DEFAULT = "gpt-3.5-turbo"

# ------------------------------------------------------------
# CSS
# ------------------------------------------------------------
st.markdown("""
<style>
.main-title {
    font-size: 42px;
    font-weight: 800;
    color: #263238;
}
.sub-title {
    font-size: 18px;
    color: #555;
}
.box {
    background-color: #f8f9fa;
    border-radius: 12px;
    padding: 18px;
    border: 1px solid #e0e0e0;
}
.metric-box {
    background-color: #eef8f5;
    padding: 15px;
    border-radius: 10px;
    text-align: center;
    border: 1px solid #b2dfdb;
}
.metric-label {
    font-size: 13px;
    color: #666;
}
.metric-value {
    font-size: 26px;
    font-weight: 700;
    color: #00897b;
}
.warning-box {
    background-color: #fff3e0;
    border-left: 5px solid #ff9800;
    padding: 12px;
    border-radius: 5px;
    font-size: 14px;
}
</style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------
# 프롬프트
# ------------------------------------------------------------
SYSTEM_PROMPT = """
당신은 Easy Read 쉬운 글 변환 전문가입니다.

목표:
어려운 문장을 발달장애인, 고령자, 외국인도 이해하기 쉬운 글로 바꿉니다.

변환 기준:
1. 한 문장에는 한 가지 정보만 담습니다.
2. 문장은 짧게 작성합니다.
3. 어려운 단어는 쉬운 말로 풀어서 설명합니다.
4. 의미는 원문과 최대한 같게 유지합니다.
5. 한자어와 외래어는 가능하면 쉬운 표현으로 바꿉니다.
6. 숫자와 비율은 이해하기 쉽게 표현합니다.

반드시 아래 JSON 형식으로만 답하세요.
설명 문장이나 마크다운은 출력하지 마세요.

{
  "converted": "쉬운 글 변환 결과",
  "summary": "핵심 내용 요약",
  "orig_grade": 3.5,
  "conv_grade": 2.0,
  "orig_easy_pct": 50,
  "conv_easy_pct": 80,
  "sentence_count_orig": 3,
  "sentence_count_conv": 5,
  "avg_sent_len_orig": 35,
  "avg_sent_len_conv": 18,
  "meaning_score": 4,
  "improvement_comment": "개선 내용 설명"
}
"""

# ------------------------------------------------------------
# 유틸 함수
# ------------------------------------------------------------
def get_openai_client(api_key):
    if not api_key:
        return None
    return OpenAI(api_key=api_key)

def parse_json_response(text):
    try:
        cleaned = text.strip()
        cleaned = re.sub(r"```json", "", cleaned)
        cleaned = re.sub(r"```", "", cleaned)
        return json.loads(cleaned), None
    except Exception as e:
        return None, str(e)

def fallback_analysis(original, converted):
    orig_sentences = max(1, len(re.findall(r"[.!?。！？\n]", original)))
    conv_sentences = max(1, len(re.findall(r"[.!?。！？\n]", converted)))

    orig_len = int(len(original) / orig_sentences)
    conv_len = int(len(converted) / conv_sentences)

    return {
        "converted": converted,
        "summary": "변환 결과를 확인하세요.",
        "orig_grade": 3.5,
        "conv_grade": 2.5,
        "orig_easy_pct": 50,
        "conv_easy_pct": 70,
        "sentence_count_orig": orig_sentences,
        "sentence_count_conv": conv_sentences,
        "avg_sent_len_orig": orig_len,
        "avg_sent_len_conv": conv_len,
        "meaning_score": 4,
        "improvement_comment": "문장을 더 짧고 쉽게 바꾸었습니다."
    }

def convert_easy_read(client, user_text, model):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"다음 글을 Easy Read 기준에 맞게 쉬운 글로 변환하세요.\n\n{user_text}"}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_tokens=1200
    )

    content = response.choices[0].message.content
    result, error = parse_json_response(content)

    if result is None:
        result = fallback_analysis(user_text, content)

    usage = response.usage
    if usage:
        result["prompt_tokens"] = usage.prompt_tokens
        result["completion_tokens"] = usage.completion_tokens
        result["total_tokens"] = usage.total_tokens
    else:
        result["prompt_tokens"] = 0
        result["completion_tokens"] = 0
        result["total_tokens"] = 0

    return result

# ------------------------------------------------------------
# 차트 함수
# ------------------------------------------------------------
def make_grade_chart(result):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["원문", "변환문"],
        y=[result["orig_grade"], result["conv_grade"]],
        text=[result["orig_grade"], result["conv_grade"]],
        textposition="auto"
    ))
    fig.update_layout(
        title="어휘 난이도 비교",
        yaxis_title="난이도 점수",
        yaxis=dict(range=[0, 5]),
        height=350
    )
    return fig

def make_easy_pct_chart(result):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=["원문", "변환문"],
        y=[result["orig_easy_pct"], result["conv_easy_pct"]],
        text=[str(result["orig_easy_pct"]) + "%", str(result["conv_easy_pct"]) + "%"],
        textposition="auto"
    ))
    fig.update_layout(
        title="쉬운 단어 비율 비교",
        yaxis_title="비율(%)",
        yaxis=dict(range=[0, 100]),
        height=350
    )
    return fig

def make_sentence_chart(result):
    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="문장 수",
        x=["원문", "변환문"],
        y=[result["sentence_count_orig"], result["sentence_count_conv"]]
    ))
    fig.add_trace(go.Bar(
        name="평균 문장 길이",
        x=["원문", "변환문"],
        y=[result["avg_sent_len_orig"], result["avg_sent_len_conv"]]
    ))
    fig.update_layout(
        title="문장 구조 비교",
        barmode="group",
        height=350
    )
    return fig

# ------------------------------------------------------------
# 사이드바
# ------------------------------------------------------------
with st.sidebar:
    st.header("⚙️ 설정")

    api_key = st.text_input(
        "OpenAI API 키",
        type="password",
        placeholder="sk-..."
    )

    model = st.selectbox(
        "사용 모델",
        ["gpt-3.5-turbo"],
        index=0
    )

    st.markdown("---")
    st.info(
        "Google Colab 계정이 아니라, 입력한 OpenAI API 키가 속한 계정의 크레딧이 사용됩니다."
    )

# ------------------------------------------------------------
# 메인 화면
# ------------------------------------------------------------
st.markdown('<div class="main-title">📖 Easy Read 변환 서비스</div>', unsafe_allow_html=True)
st.markdown(
    '<div class="sub-title">LLM 프롬프트 엔지니어링 기반 쉬운 글 변환 대시보드</div>',
    unsafe_allow_html=True
)

st.markdown("---")

st.markdown("""
<div class="warning-box">
해당 프로젝트는 단순한 프롬프트 입력이 아니라, 프롬프트 엔지니어링을 활용하여
어려운 문서를 쉬운 글로 변환하고 그 결과를 시각화하는 LLM 활용 서비스입니다.
</div>
""", unsafe_allow_html=True)

st.markdown("")

col1, col2 = st.columns(2)

with col1:
    st.subheader("📄 원문 입력")
    user_text = st.text_area(
        "변환할 문장을 입력하세요.",
        height=300,
        placeholder="예시: 국민기초생활 보장법에 의거하여 수급자로 선정된 가구는 생계급여 지급 기준을 충족하여야 합니다."
    )

with col2:
    st.subheader("✅ 변환 결과")
    result_area = st.empty()

run_button = st.button("🚀 Easy Read 변환 실행", type="primary")

if run_button:
    if not api_key:
        st.error("OpenAI API 키를 입력해야 합니다.")
    elif not user_text.strip():
        st.error("변환할 원문을 입력해야 합니다.")
    else:
        client = get_openai_client(api_key)

        with st.spinner("GPT-3.5-turbo 모델로 변환 중입니다..."):
            try:
                result = convert_easy_read(client, user_text, model)
                st.session_state["result"] = result
                st.session_state["original"] = user_text
                st.success("변환이 완료되었습니다.")
            except Exception as e:
                st.error(f"오류 발생: {e}")

# ------------------------------------------------------------
# 결과 출력
# ------------------------------------------------------------
if "result" in st.session_state:
    result = st.session_state["result"]
    original = st.session_state["original"]

    with col2:
        result_area.text_area(
            "쉬운 글 변환 결과",
            value=result["converted"],
            height=300
        )

    st.markdown("---")
    st.subheader("📊 핵심 평가 지표")

    m1, m2, m3, m4 = st.columns(4)

    grade_improve = round(result["orig_grade"] - result["conv_grade"], 2)
    easy_improve = result["conv_easy_pct"] - result["orig_easy_pct"]

    with m1:
        st.markdown(
            f"""
            <div class="metric-box">
                <div class="metric-label">어휘 난이도 개선</div>
                <div class="metric-value">▼ {grade_improve}</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with m2:
        st.markdown(
            f"""
            <div class="metric-box">
                <div class="metric-label">쉬운 단어 비율</div>
                <div class="metric-value">{result["conv_easy_pct"]}%</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with m3:
        st.markdown(
            f"""
            <div class="metric-box">
                <div class="metric-label">의미 보존 점수</div>
                <div class="metric-value">{result["meaning_score"]}/5</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    with m4:
        st.markdown(
            f"""
            <div class="metric-box">
                <div class="metric-label">총 사용 토큰</div>
                <div class="metric-value">{result["total_tokens"]}</div>
            </div>
            """,
            unsafe_allow_html=True
        )

    st.markdown("---")
    st.subheader("📈 변환 전후 시각화")

    c1, c2 = st.columns(2)

    with c1:
        st.plotly_chart(make_grade_chart(result), use_container_width=True)

    with c2:
        st.plotly_chart(make_easy_pct_chart(result), use_container_width=True)

    st.plotly_chart(make_sentence_chart(result), use_container_width=True)

    st.markdown("---")
    st.subheader("📋 변환 분석 요약")

    analysis_df = pd.DataFrame({
        "항목": [
            "원문 어휘 난이도",
            "변환문 어휘 난이도",
            "원문 쉬운 단어 비율",
            "변환문 쉬운 단어 비율",
            "원문 문장 수",
            "변환문 문장 수",
            "원문 평균 문장 길이",
            "변환문 평균 문장 길이",
            "의미 보존 점수",
            "프롬프트 토큰",
            "응답 토큰",
            "총 토큰"
        ],
        "값": [
            result["orig_grade"],
            result["conv_grade"],
            str(result["orig_easy_pct"]) + "%",
            str(result["conv_easy_pct"]) + "%",
            result["sentence_count_orig"],
            result["sentence_count_conv"],
            str(result["avg_sent_len_orig"]) + "자",
            str(result["avg_sent_len_conv"]) + "자",
            str(result["meaning_score"]) + "/5",
            result["prompt_tokens"],
            result["completion_tokens"],
            result["total_tokens"]
        ]
    })

    st.dataframe(analysis_df, use_container_width=True, hide_index=True)

    st.markdown("---")
    st.subheader("📝 보고서용 문장")

    report_text = f"""
프로젝트는 프롬프트 엔지니어링을 활용하여 어려운 문서를 쉬운 글로 변환하는 LLM 기반 서비스이다.
사용 모델은 수업 기준과 토큰 비용 절감을 고려하여 gpt-3.5-turbo를 사용하였다.

시스템 프롬프트에는 짧은 문장 사용, 쉬운 단어 선택, 어려운 용어 풀어쓰기, 의미 보존, JSON 형식 출력 조건을 포함하였다.
이를 통해 LLM이 단순히 문장을 바꾸는 것이 아니라 Easy Read 기준에 맞게 결과를 생성하도록 설계하였다.

변환 결과는 원문과 비교하여 어휘 난이도, 쉬운 단어 비율, 문장 수, 평균 문장 길이, 의미 보존 점수로 평가하였다.
이번 실행에서는 어휘 난이도가 {result["orig_grade"]}에서 {result["conv_grade"]}로 변화하였고,
쉬운 단어 비율은 {result["orig_easy_pct"]}%에서 {result["conv_easy_pct"]}%로 변화하였다.
총 사용 토큰 수는 {result["total_tokens"]}개이다.

따라서 본 서비스는 프롬프트 설계, LLM API 활용, 결과 시각화, 성능평가를 결합한 프롬프트 엔지니어링 기반 Easy Read 변환 서비스라고 볼 수 있다.
"""

    st.text_area(
        "보고서에 붙여넣기",
        value=report_text.strip(),
        height=260
    )

    st.download_button(
        "📥 변환 결과 다운로드",
        data=f"[원문]\n{original}\n\n[쉬운 글 변환 결과]\n{result['converted']}\n\n[분석 요약]\n{report_text}",
        file_name="easy_read_result.txt",
        mime="text/plain"
    )
'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py 생성 완료")


# 3. localtunnel 설치
!npm install -g localtunnel

# 4. 기존 Streamlit 종료
!pkill -f streamlit

# 5. Streamlit 백그라운드 실행
!streamlit run /content/app.py --server.port 8501 --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

# 6. 외부 접속 링크 생성
!npx localtunnel --port 8501

app.py 생성 완료
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴⠙⠹⠸⠼⠴⠦your url is: https://social-clocks-sip.loca.lt
^C


In [14]:
!pkill -f streamlit

!streamlit run /content/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  > /content/streamlit.log 2>&1 &

from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>